# Data Information

This dataset contains medical insurance cost information for 1338 individuals. It includes demographic and health-related variables such as age, sex, BMI, number of children, smoking status, and residential region in the US. The target variable is charges, which represents the medical insurance cost billed to the individual.

Source: https://doi.org/10.34740/kaggle/dsv/12853160

# Dataset

In [616]:
import pandas as pd

data_df = pd.read_csv("insurance.csv")
data_df.head(10)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


# Problem Definition

## Objective

- use the dataset to estimate insurance bill/cost per person
- identify which variables are most associated with higher or lower insurance bill

## Usecase

Corporate Wellness Program
- Identify high-risk employees based on predicted healthcare costs
- Enable targeted interventions to reduce the company’s overall healthcare expenses

Insurance Company
- Analyze healthcare cost trends across different population segments
- Enable dynamic premium pricing based on individual risk profiles
- Provide personalized insurance plans tailored to each customer

Example Use Case
A person with certain conditions who is predicted to have high healthcare costs can be recommended lifestyle interventions

# Data Preparation

## 1. Data Understanding

In [617]:
print("Shape:", data_df.shape)

Shape: (1338, 7)


In [618]:
print("Columns:", data_df.columns)

Columns: Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='str')


In [619]:
data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [620]:
data_df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [621]:
data_df.duplicated().sum()

np.int64(1)

In [622]:
data_df = data_df.drop_duplicates()
data_df.duplicated().sum()

np.int64(0)

In [623]:
data_df.describe().round(2)

,age,bmi,children,charges
count,1337.00,1337.00,1337.00,1337.00
mean,39.22,30.66,1.10,13279.12
std,14.04,6.10,1.21,12110.36
min,18.00,15.96,0.00,1121.87
25%,27.00,26.29,0.00,4746.34
50%,39.00,30.40,1.00,9386.16
75%,51.00,34.70,2.00,16657.72
max,64.00,53.13,5.00,63770.43


In [624]:
data_df.describe(include="str")

,sex,smoker,region
count,1337,1337,1337
unique,2,2,4
top,male,no,southeast
freq,675,1063,364


# 2. Data Cleaning

## 2.1 Outlier 

In [625]:
num_cols = ['age', 'bmi', 'charges']

for col in num_cols:
    Q1 = data_df[col].quantile(0.25)
    Q3 = data_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = data_df[(data_df[col] < lower) | (data_df[col] > upper)]
    
    print(f"{col}: {len(outliers)} outliers")

age: 0 outliers
bmi: 9 outliers
charges: 139 outliers


The BMI outlier in this dataset is above ~47.3 (upper bound IQR), with a max of 53.13. This value is included in the Obesity Class III category, which medically actually exists and is documented. This is not a data entry error.

Outlier charges are retained because they are the target variable that the model must predict.

In [626]:
data_df['children'].value_counts()

children
0    573
1    324
2    240
3    157
4     25
5     18
Name: count, dtype: int64

In [627]:
data_df['region'].value_counts()

region
southeast    364
southwest    325
northwest    324
northeast    324
Name: count, dtype: int64

In [628]:
data_df['sex'].value_counts()

sex
male      675
female    662
Name: count, dtype: int64

In [629]:
data_df['smoker'].value_counts()

smoker
no     1063
yes     274
Name: count, dtype: int64

# 3. Data Transformation

## 3.1 Feature Encoding

In [630]:
data_df['gender_encoded'] = data_df['sex'].map({'male': 0, 'female': 1})
data_df.head(5)

,age,sex,bmi,children,smoker,region,charges,gender_encoded
0,19,female,27.900,0,yes,southwest,16884.92400,1
1,18,male,33.770,1,no,southeast,1725.55230,0
2,28,male,33.000,3,no,southeast,4449.46200,0
3,33,male,22.705,0,no,northwest,21984.47061,0
4,32,male,28.880,0,no,northwest,3866.85520,0


In [631]:
data_df['smoker_encoded'] = data_df['smoker'].map({'no': 0, 'yes': 1})
data_df.head(5)

,age,sex,bmi,children,smoker,region,charges,gender_encoded,smoker_encoded
0,19,female,27.900,0,yes,southwest,16884.92400,1,1
1,18,male,33.770,1,no,southeast,1725.55230,0,0
2,28,male,33.000,3,no,southeast,4449.46200,0,0
3,33,male,22.705,0,no,northwest,21984.47061,0,0
4,32,male,28.880,0,no,northwest,3866.85520,0,0


In [632]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' opsional

region_encoded = encoder.fit_transform(data_df[['region']])

region_columns = encoder.get_feature_names_out(['region'])
region_df = pd.DataFrame(region_encoded, columns=region_columns, index=data_df.index)

data_df = data_df.drop('region', axis=1)
data_df = pd.concat([data_df, region_df], axis=1)

data_df.head(5)

,age,sex,bmi,children,smoker,charges,gender_encoded,smoker_encoded,region_northwest,region_southeast,region_southwest
0,19,female,27.900,0,yes,16884.92400,1,1,0.0,0.0,1.0
1,18,male,33.770,1,no,1725.55230,0,0,0.0,1.0,0.0
2,28,male,33.000,3,no,4449.46200,0,0,0.0,1.0,0.0
3,33,male,22.705,0,no,21984.47061,0,0,1.0,0.0,0.0
4,32,male,28.880,0,no,3866.85520,0,0,1.0,0.0,0.0


In [633]:
data_df = data_df.drop(columns=['sex', 'smoker'])
data_df.head(5)

,age,bmi,children,charges,gender_encoded,smoker_encoded,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,1,1,0.0,0.0,1.0
1,18,33.770,1,1725.55230,0,0,0.0,1.0,0.0
2,28,33.000,3,4449.46200,0,0,0.0,1.0,0.0
3,33,22.705,0,21984.47061,0,0,1.0,0.0,0.0
4,32,28.880,0,3866.85520,0,0,1.0,0.0,0.0


# Data Analysis

## 1. Descriptive Statistics

In [634]:
data_df[['age', 'bmi', 'children', 'charges']].describe().round(2)

,age,bmi,children,charges
count,1337.00,1337.00,1337.00,1337.00
mean,39.22,30.66,1.10,13279.12
std,14.04,6.10,1.21,12110.36
min,18.00,15.96,0.00,1121.87
25%,27.00,26.29,0.00,4746.34
50%,39.00,30.40,1.00,9386.16
75%,51.00,34.70,2.00,16657.72
max,64.00,53.13,5.00,63770.43


## 2. Inferential Statistics

## 2.1 Normality Test

In [635]:
from scipy import stats

stat, p = stats.shapiro(data_df['charges'])
print(f"Shapiro-Wilk: stat={stat:.4f}, p-value={p:.4e}")
print(f"Skewness: {data_df['charges'].skew():.2f}")

Shapiro-Wilk: stat=0.8148, p-value=1.1960e-36
Skewness: 1.52


Because charges are highly skewed (skewness = 1.52) and the Shapiro-Wilk test rejects normality (p < 0.05), we use non-parametric tests.

## 2.2 Binary Features VS Charges (Mann-Whitney U Test)

In [636]:
from scipy.stats import mannwhitneyu

for col in ['smoker_encoded', 'gender_encoded']:
    group_0 = data_df[data_df[col] == 0]['charges']
    group_1 = data_df[data_df[col] == 1]['charges']
    stat, p = mannwhitneyu(group_0, group_1, alternative='two-sided')
    print(f"{col}: U={stat:.2f}, p-value={p:.4e} --> {'Significant' if p < 0.05 else 'Not Significant'}")

smoker_encoded: U=7403.00, p-value=5.7470e-130 --> Significant
gender_encoded: U=226198.00, p-value=6.9448e-01 --> Not Significant


## 2.3 Multi-group Features vs Charges (Kruskal-Wallis Test)

In [637]:
from scipy.stats import kruskal

# Region
data_df['region_label'] = data_df[['region_northwest','region_southeast','region_southwest']].idxmax(axis=1)
data_df.loc[(data_df[['region_northwest','region_southeast','region_southwest']].sum(axis=1) == 0), 'region_label'] = 'region_northeast'

region_groups = [group['charges'].values for name, group in data_df.groupby('region_label')]
stat_r, p_r = kruskal(*region_groups)
print(f"region: H={stat_r:.2f}, p-value={p_r:.4e} --> {'Significant' if p_r < 0.05 else 'Not Significant'}")

# Children
children_groups = [group['charges'].values for name, group in data_df.groupby('children')]
stat_c, p_c = kruskal(*children_groups)
print(f"children: H={stat_c:.2f}, p-value={p_c:.4e} --> {'Significant' if p_c < 0.05 else 'Not Significant'}")

data_df = data_df.drop(columns=['region_label'])

region: H=4.62, p-value=2.0162e-01 --> Not Significant
children: H=29.12, p-value=2.1957e-05 --> Significant


## 2.4 Continuous Features vs Charges (Spearman Correlation)

In [638]:
from scipy.stats import spearmanr

for col in ['age', 'bmi']:
    corr, p = spearmanr(data_df[col], data_df['charges'])
    print(f"{col}: rho={corr:.4f}, p-value={p:.4e}")

age: rho=0.5335, p-value=3.1878e-99
bmi: rho=0.1196, p-value=1.1637e-05


### Inferential Statistics Summary

| Feature | Test | Statistic | p-value | Result |
|---------|------|-----------|---------|--------|
| smoker | Mann-Whitney U | U = 7,403.00 | 5.75e-130 | **Significant** |
| gender | Mann-Whitney U | U = 226,198.00 | 6.94e-01 | Not Significant |
| region | Kruskal-Wallis | H = 4.62 | 2.02e-01 | Not Significant |
| children | Kruskal-Wallis | H = 29.12 | 2.20e-05 | **Significant** |
| age | Spearman | rho = 0.5335 | 3.19e-99 | **Significant** (moderate +) |
| bmi | Spearman | rho = 0.1196 | 1.16e-05 | **Significant** (weak +) |

**Key Findings:**
- **Smoker** is the strongest predictor of charges by far (extremely low p-value)
- **Age** has a moderate positive correlation with charges -- older individuals tend to have higher costs
- **Children** has a significant effect -- the number of dependents influences insurance costs
- **BMI** has a weak but statistically significant positive correlation with charges
- **Gender** and **Region** have no significant effect on charges